In [1]:
import pandas as pd
import numpy as np

# =====================
# 1. Load data
# =====================
df = pd.read_csv("osc_sheet_11_with_industry_score.csv")

print("Original shape:", df.shape)

# =====================
# 2. Define target
# =====================
price_col = "Closing Price.1"

# Create binary target
# Investable (1): price >= median
# Non-investable (0): price < median
threshold = 8.0 # percentage threshold
df["investable"] = ((df[price_col] - df["Closing Price"]) / df[price_col] * 100 >= threshold).astype(int)

# =====================
# 3. Feature / target split
# =====================
X = df.drop(columns=[
    price_col,
    "investable",
    "Company Name",
    "ISIN code",
    "NSE symbol",
    "Date",
    "Date.1",
    "Industry group"
], errors="ignore")

y = df["investable"]

print("Features shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

# =====================
# 4. Handle missing values
# =====================
# Random Forest cannot handle NaNs
X = X.fillna(X.median(numeric_only=True))

# =====================
# 5. Train-test split
# =====================
from sklearn.model_selection import train_test_split
df["split"] = "unassigned"
df["y_true"] = df["investable"]
df["rf_pred"] = np.nan
df["rf_prob"] = np.nan
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
train_idx = X_train.index
test_idx = X_test.index

# =====================
# 6. Train Random Forest
# =====================
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
   n_estimators=500,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight={0: 1, 1: 3}
)

rf.fit(X_train, y_train)

# =====================
# 7. Evaluation
# =====================
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]
y_prob_randoforest=y_prob

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# =====================
# 8. Feature importance
# =====================
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 15 Important Features:")
print(feature_importance.head(15))



Original shape: (998, 30)
Features shape: (998, 23)
Target distribution:
investable
0    717
1    281
Name: count, dtype: int64

Accuracy: 0.77
ROC-AUC: 0.8356894841269842

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.92      0.85       144
           1       0.65      0.39      0.49        56

    accuracy                           0.77       200
   macro avg       0.72      0.65      0.67       200
weighted avg       0.75      0.77      0.75       200


Confusion Matrix:
[[132  12]
 [ 34  22]]

Top 15 Important Features:
Closing Price                                   0.186066
Shares Outstanding                              0.075235
EPS                                             0.057667
P/E                                             0.048273
Earnings per share before extraordinary item    0.046541
Market Capitalisation                           0.040782
Enterprise value                                0.039468
Interest

In [2]:
df.loc[train_idx, "split"] = "train"
df.loc[test_idx, "split"] = "test"
y_test_pred = rf.predict(X_test)
y_test_prob = rf.predict_proba(X_test)[:, 1]

df.loc[test_idx, "rf_pred"] = y_test_pred
df.loc[test_idx, "rf_prob"] = y_test_prob
df.to_csv("osc_sheet_11_with_split_info.csv", index=False)
print("Saved CSV with train-test split info")


Saved CSV with train-test split info


In [3]:
import pandas as pd
COMPANY_COL="Company Name"
# --- Train table ---
train_df = pd.DataFrame({
    'Company': df.loc[X_train.index, COMPANY_COL],
    'Set': 'Train',
    'Investable': y_train.values
})

# --- Test table ---
test_df = pd.DataFrame({
    'Company': df.loc[X_test.index, COMPANY_COL],
    'Set': 'Test',
    'Investable': y_test.values
})

# Combine
full_df = pd.concat([train_df, test_df], ignore_index=True)

# Sort nicely
full_df = full_df.sort_values(by=['Set', 'Investable'], ascending=[True, False])

print(full_df)



                                  Company    Set  Investable
799      Happiest Minds Technologies Ltd.   Test           1
800                  G T P L Hathway Ltd.   Test           1
801       Honeywell Automation India Ltd.   Test           1
804             Polyplex Corporation Ltd.   Test           1
805  Tata Teleservices (Maharashtra) Ltd.   Test           1
..                                    ...    ...         ...
792                    Dalmia Bharat Ltd.  Train           0
793               Varroc Engineering Ltd.  Train           0
795                        Redington Ltd.  Train           0
796        Mahanagar Telephone Nigam Ltd.  Train           0
797              Thomas Cook (India) Ltd.  Train           0

[998 rows x 3 columns]


In [4]:
X_no_price = X.drop(columns=['Closing Price'])

Xtr, Xte, ytr, yte = train_test_split(
    X_no_price, y, test_size=0.2, random_state=42
)

rf.fit(Xtr, ytr)
print("Accuracy (no price):", accuracy_score(yte, rf.predict(Xte)))
print("ROC-AUC (no price):", roc_auc_score(yte, rf.predict_proba(Xte)[:,1]))


Accuracy (no price): 0.64
ROC-AUC (no price): 0.6620988099531194


In [8]:
import pandas as pd
import numpy as np

# =====================
# 1. Load data
# =====================
df = pd.read_csv("osc_sheet_11_with_industry_score.csv")

print("Original shape:", df.shape)

# =====================
# 2. Define target
# =====================
price_col = "Closing Price.1"

threshold = df[price_col].median()
df["investable"] = (df[price_col] >= threshold).astype(int)

# =====================
# 3. Feature / target split
# =====================
X = df.drop(columns=[
    price_col,
    "investable",
    "Company Name",
    "ISIN code",
    "NSE symbol",
    "Date",
    "Date.1",
    "Industry group"
], errors="ignore")

y = df["investable"]

print("Features shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

# =====================
# 4. Handle missing values
# =====================
X = X.fillna(X.median(numeric_only=True))

# =====================
# 5. Train-test split
# =====================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================
# 6. Feature scaling (CRITICAL)
# =====================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =====================
# 7. Train Logistic Regression
# =====================
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)




log_reg.fit(X_train_scaled, y_train)

# =====================
# 8. Evaluation
# =====================
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_pred = log_reg.predict(X_test_scaled)
y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]
y_prob_logistic=y_prob

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# =====================
# 9. Coefficients (Feature importance for Logistic Regression)
# =====================
coef = pd.Series(
    log_reg.coef_[0],
    index=X.columns
).sort_values(key=np.abs, ascending=False)

print("\nTop 15 Influential Features (by |coefficient|):")
print(coef.head(15))


Original shape: (998, 30)
Features shape: (998, 23)
Target distribution:
investable
1    499
0    499
Name: count, dtype: int64

Accuracy: 0.86
ROC-AUC: 0.9375

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.88      0.86       100
           1       0.88      0.84      0.86       100

    accuracy                           0.86       200
   macro avg       0.86      0.86      0.86       200
weighted avg       0.86      0.86      0.86       200


Confusion Matrix:
[[88 12]
 [16 84]]

Top 15 Influential Features (by |coefficient|):
Closing Price                                                              4.663771
Market Capitalisation                                                      4.531892
Shares Outstanding                                                        -3.586877
Earnings per share before extraordinary item                               1.776389
Enterprise value                                                  

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.14/

In [6]:
# Predict probabilities


# Combine probabilities
ensemble_prob = 0.4* y_prob_randoforest + 0.6* y_prob_logistic

# Final prediction
ensemble_pred = (ensemble_prob >= 0.5).astype(int)

print("Ensemble Accuracy:", accuracy_score(y_test, ensemble_pred))
print("Ensemble ROC-AUC:", roc_auc_score(y_test, ensemble_prob))
print("\nClassification Report:")
print(classification_report(y_test, ensemble_pred))

Ensemble Accuracy: 0.85
Ensemble ROC-AUC: 0.9317

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.91      0.86       100
           1       0.90      0.79      0.84       100

    accuracy                           0.85       200
   macro avg       0.86      0.85      0.85       200
weighted avg       0.86      0.85      0.85       200

